In [15]:
import os
import re
import pandas as pd

ROOT_DIR = r"BenchmarkJava-master/src/main/java/org/owasp/benchmark/testcode"

VULN_MAP = {
    "pathtraver": "Path Traversal",
    "sqli": "SQL Injection",
    "cmdi": "Command Injection",
    "ldapi": "LDAP Injection",
    "xss": "Cross Site Scripting",
    "trustbound": "Trust Boundary Violation",
    "weakrand": "Weak Randomness",
    "crypto": "Weak Cryptography",
    "hash": "Weak Hashing",
    "securecookie": "Insecure Cookie Handling",
    "xpathi": "XPath Injection"
}

records = []

for root, dirs, files in os.walk(ROOT_DIR):

    for file in files:

        if not file.endswith(".java"):
            continue

        path = os.path.join(root, file)

        try:

            with open(path, "r", encoding="utf-8") as f:
                code = f.read()

            servlet_path = ""

            servlet_match = re.search(
                r'@WebServlet\s*\(\s*value\s*=\s*"([^"]+)"',
                code,
                re.DOTALL
            )

            if servlet_match:
                servlet_path = servlet_match.group(1)

            vulnerability = "Unknown"

            for key, value in VULN_MAP.items():

                if key in servlet_path.lower():
                    vulnerability = value
                    break

            package_match = re.search(
                r'package\s+([a-zA-Z0-9_.]+);',
                code
            )

            package_name = ""

            if package_match:
                package_name = package_match.group(1)

            records.append({
                "file_name": file,
                "package": package_name,
                "servlet_path": servlet_path,
                "vulnerability": vulnerability,
                "source_code": code
            })

        except Exception as e:

            print(f"ERROR: {file}")
            print(e)

df = pd.DataFrame(records)

df = df.drop_duplicates(
    subset=["file_name"]
)

df.to_csv(
    "owasp_benchmark_dataset.csv",
    index=False
)

print("\nDATASET CREATED")
print("Rows:", len(df))

print("\nVulnerability Distribution:")
print(
    df["vulnerability"]
    .value_counts()
)


DATASET CREATED
Rows: 2740

Vulnerability Distribution:
vulnerability
SQL Injection               504
Weak Randomness             493
Cross Site Scripting        455
Path Traversal              268
Command Injection           251
Weak Cryptography           246
Weak Hashing                236
Trust Boundary Violation    126
Insecure Cookie Handling     67
LDAP Injection               59
XPath Injection              35
Name: count, dtype: int64


In [21]:
import pandas as pd

df = pd.read_csv(
    "BenchmarkJava-master/scorecard/Benchmark_v1.2_Scorecard_for_SonarQube_Java_Plugin.csv"
)

print(len(df))
print(df.head())
print(df.tail())

2740
          # test name     category   CWE  real vulnerability  \
0  BenchmarkTest00001   pathtraver    22                true   
1  BenchmarkTest00002   pathtraver    22                true   
2  BenchmarkTest00003         hash   328                true   
3  BenchmarkTest00004   trustbound   501                true   
4  BenchmarkTest00005       crypto   327                true   

   identified by tool  pass/fail   Benchmark version: 1.2  \
0               false       fail                      NaN   
1               false       fail                      NaN   
2               false       fail                      NaN   
3               false       fail                      NaN   
4                true       pass                      NaN   

    Actual results generated: 2016-09-19  
0                                    NaN  
1                                    NaN  
2                                    NaN  
3                                    NaN  
4                           

In [23]:
import pandas as pd

owasp_df = pd.read_csv(
    "owasp_benchmark_dataset.csv"
)

scorecard_df = pd.read_csv(
    "BenchmarkJava-master/scorecard/Benchmark_v1.2_Scorecard_for_SonarQube_Java_Plugin.csv",
    comment="#",
    header=None,
    names=[
        "test_name",
        "category",
        "cwe",
        "real_vulnerability",
        "identified_by_tool",
        "pass_fail"
    ]
)

scorecard_df["file_name"] = (
    scorecard_df["test_name"] + ".java"
)

merged = owasp_df.merge(
    scorecard_df[
        [
            "file_name",
            "category",
            "cwe",
            "real_vulnerability"
        ]
    ],
    on="file_name",
    how="inner"
)

print("Merged rows:", len(merged))

merged.to_csv(
    "owasp_security_dataset.csv",
    index=False
)

Merged rows: 2740


In [28]:
import pandas as pd

df = pd.read_csv("owasp_security_dataset.csv")

df["real_vulnerability"] = (
    df["real_vulnerability"]
    .astype(str)
    .str.strip()
    .str.lower()
)

vuln_df = df[
    df["real_vulnerability"] == "true"
]

print("Vulnerable samples:", len(vuln_df))
print("Non-vulnerable samples:",
      len(df[df["real_vulnerability"] == "false"]))

vuln_df.to_csv(
    "vulnerable_only.csv",
    index=False
)

Vulnerable samples: 1415
Non-vulnerable samples: 1325


In [30]:
import pandas as pd

df = pd.read_csv("vulnerable_only.csv")

cols = [
    "file_name",
    "category",
    "cwe",
    "source_code"
]

df = df[cols]

df.to_csv(
    "vulnerable_training_pool.csv",
    index=False
)

print(len(df))

1415


In [31]:
import pandas as pd

df = pd.read_csv(
    "vulnerable_training_pool.csv"
)

sample_df = (
    df.groupby("category", group_keys=False)
      .apply(
          lambda x: x.sample(
              min(50, len(x)),
              random_state=42
          )
      )
)

sample_df.to_csv(
    "fix_generation_batch.csv",
    index=False
)

print(len(sample_df))

478


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_15916\1539717997.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [33]:
import pandas as pd
import os

df = pd.read_csv("owasp_security_dataset.csv")

df["real_vulnerability"] = (
    df["real_vulnerability"]
    .astype(str)
    .str.strip()
    .str.lower()
)

vuln_df = df[df["real_vulnerability"] == "true"]

os.makedirs("sample_vulnerabilities", exist_ok=True)

for category in vuln_df["category"].unique():

    sample = vuln_df[
        vuln_df["category"] == category
    ].head(1)

    if len(sample) == 0:
        continue

    row = sample.iloc[0]

    filename = f"sample_vulnerabilities/{category}.java"

    with open(filename, "w", encoding="utf-8") as f:
        f.write(row["source_code"])

    print("Saved:", filename)

Saved: sample_vulnerabilities/ pathtraver.java
Saved: sample_vulnerabilities/ hash.java
Saved: sample_vulnerabilities/ trustbound.java
Saved: sample_vulnerabilities/ crypto.java
Saved: sample_vulnerabilities/ cmdi.java
Saved: sample_vulnerabilities/ sqli.java
Saved: sample_vulnerabilities/ ldapi.java
Saved: sample_vulnerabilities/ xss.java
Saved: sample_vulnerabilities/ weakrand.java
Saved: sample_vulnerabilities/ securecookie.java
Saved: sample_vulnerabilities/ xpathi.java


In [ ]:
import pandas as pd

df = pd.read_csv("vulnerable_only.csv")

print(
    df["category"]
    .value_counts()
)

category
sqli            272
xss             246
weakrand        218
pathtraver      133
crypto          130
hash            129
cmdi            126
trustbound       83
securecookie     36
ldapi            27
xpathi           15
Name: count, dtype: int64


In [35]:
import pandas as pd

df = pd.read_csv("vulnerable_only.csv")

sample_df = (
    df.groupby("category", group_keys=False)
      .apply(lambda x: x.sample(min(10, len(x)), random_state=42))
)

sample_df.to_csv(
    "remediation_pilot_100.csv",
    index=False
)

print("Samples:", len(sample_df))

Samples: 110


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_15916\4151603383.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(10, len(x)), random_state=42))


In [36]:
import pandas as pd
import json

df = pd.read_csv("remediation_pilot_100.csv")

with open("fix_generation_prompts.jsonl", "w", encoding="utf-8") as f:

    for _, row in df.iterrows():

        prompt = f"""
You are a senior Java security engineer.

CWE: CWE-{row['cwe']}
Category: {row['category']}

Task:
Fix the vulnerability.

Return:
1. Fixed Java code
2. Explanation of the fix

Java Code:

{row['source_code']}
"""

        record = {
            "file_name": row["file_name"],
            "prompt": prompt
        }

        f.write(json.dumps(record) + "\n")

print("Done")

Done


In [37]:
import pandas as pd

df = pd.read_csv("vulnerable_only.csv")

summary = (
    df.groupby(["category", "cwe"])
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
)

print(summary)

summary.to_csv(
    "vulnerability_summary.csv",
    index=False
)

         category  cwe  count
6            sqli   89    272
10            xss   79    246
8        weakrand  330    218
4      pathtraver   22    133
1          crypto  327    130
2            hash  328    129
0            cmdi   78    126
7      trustbound  501     83
5    securecookie  614     36
3           ldapi   90     27
9          xpathi  643     15


In [38]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("vulnerable_only.csv")

train_df, eval_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df["category"]
)

train_df.to_csv("train_vulnerable.csv", index=False)
eval_df.to_csv("eval_vulnerable.csv", index=False)

print("Train:", len(train_df))
print("Eval :", len(eval_df))

Train: 1202
Eval : 213


In [ ]:
import pandas as pd
import json
from openai import OpenAI

client = OpenAI(api_key="YOUR_API_KEY")

df = pd.read_csv("train_vulnerable.csv")

df = df.head(50)

with open("training_pairs.jsonl", "w", encoding="utf-8") as outfile:

    for idx, row in df.iterrows():

        prompt = f"""
You are a senior Java security engineer.

Vulnerability Category: {row['category']}
CWE: CWE-{row['cwe']}

Task:
Fix the vulnerability while preserving functionality.

Return ONLY the complete fixed Java code.

Java Code:
{row['source_code']}
"""

        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        fixed_code = response.choices[0].message.content

        record = {
            "instruction": f"Fix CWE-{row['cwe']} {row['category']} vulnerability",
            "input": row["source_code"],
            "output": fixed_code
        }

        outfile.write(json.dumps(record) + "\n")

        print(f"Processed {idx+1}/50")